# 🔐 **Security Evaluation of a Face Recognition System**

## Adversarial Attacks Evaluation on NN1

**Academic Year:** 2024-2025  
**Group:** 04

---

### 👥 Team Members
- **Agostino Cardamone** — `0622702276`
- **Asja Antonucci**     — `0622702437`
- **Chiara Ferraioli**   — `0622702169`

---

### 📚 Overview of This Section

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Generating Adversarial Examples](#2-generating-adversarial-examples)

## 1. Setup and Data Loading

#### Environment Setup

To ensure reproducibility and avoid package conflicts, it is strongly recommended to run all experiments in an isolated environment. We use Conda to create and manage the project environment, and all Python dependencies are listed in the requirements.txt file.

In [ ]:
# 1) Create a new environment named “aic_env”
#conda create -n aic_env python=3.10 -y

# 2) Switch into the new environment
# On Windows:
# conda activate aic_env
# On Linux/macOS:
# source activate aic_env

# 3) Install all dependencies
#!pip install -r requirements.txt

import os 

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("torch.version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

#### Dataset Configuration and Paths

This section defines the core paths and settings used throughout the project to manage the dataset structure and preprocessing behavior.

- `dataset_dir` points to the root directory containing the dataset files.
- `dataset_selection` controls whether to regenerate the test set from scratch (⚠️set to True only if you have extracted `vggface2_train` inside `dataset/vggface2_train/trainset`⚠️)
- `mtcnn_processing_nn1` determines whether to apply MTCNN face alignment for NN1 preprocessing.
- `test_set_rnd` specifies whether the test set should be built randomly or from the pre-defined class list in `test_set.csv`.

Metadata and folder structure:
- `vgg2_dataset_annotations_path` points to `identity_meta.csv`, which contains class metadata (name, gender, etc.).
- `test_set_data_folder` is the location of test samples.
- `test_set_annotations_folder` contains the CSV file describing the test set structure.

All experiment outputs will be saved to:
- `results_folder` — for accuracy results and evaluations.
- `adversarial_folder` — for storing generated adversarial images.

The variable `device` automatically selects GPU if available, otherwise defaults to CPU.

In [ ]:
from utils import *         # Project-specific utilities and imports

# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base directory containing all dataset-related files
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# If True, a new test set will be built by sampling and copying images from the original VGGFace2 dataset
# NOTE: This requires the dataset to be downloaded and extracted under 'vggface2_train/trainset'
# If False, the existing CSV files will be loaded without modifying or copying any images
dataset_selection = False  

# If True, test images will be aligned and cropped using MTCNN preprocessing (for NN1 compatibility)
mtcnn_processing_nn1 = False

# If True, the test set will be built via random sampling of identities and images
# If False, the test set will be built based on predefined class IDs listed in 'test_set.csv'
test_set_rnd = False

# Path to VGGFace2 identity metadata (includes Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set files and images
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')

# Directory to store evaluation results (e.g., SEC curves)
results_folder = os.path.join(os.getcwd(), 'results')

# Directory to save generated adversarial examples
adversarial_folder = os.path.join(os.getcwd(), 'attacks')

# Device configuration: use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


#### Load Test Set and Class Labels

This block performs two critical initializations:

1. **Load test set metadata**
   - The file `test_set.csv` is read into a DataFrame.
   - It contains exactly 100 test identities, each with 10 face images located in `testset/samples/`.
   - The total number of expected images is computed as `100 × 10 = 1000`.

2. **Load class label mappings**
   - The face recognition model requires access to the full list of class names (8631 identities).
   - These are loaded from a `.npy` file originally published by the official [`rcmalli/keras-vggface`](https://github.com/rcmalli/keras-vggface) repository.
   - If the file is not present locally, it is automatically downloaded.
   - Any surrounding whitespace in label strings is stripped to ensure clean formatting.

This step is necessary to convert model outputs (class indices) into readable identity labels.

In [ ]:
# —————————————————————————————————————————————
#              Load annotation CSVs
# —————————————————————————————————————————————

# Read the existing test_set.csv describing our 100 test identities
# (each identity will have 10 samples in the folder structure)
test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# Calculate how many total images we expect in the test set:
# number of identities × 10 images each
test_set_size = len(test_set) * 10

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

#### Load and Prepare Face Recognition Model (NN1)

This section sets up the face recognition model referred to as **NN1**, based on the `InceptionResnetV1` architecture provided by the `facenet-pytorch` library.

- The model is initialised with weights pre-trained on the **VGGFace2** dataset, which contains over 8,000 people identities.
- It is set to evaluation mode (`.eval()`), disabling stochastic layers such as dropout and batch normalisation updates, ensuring deterministic inference.
- By default, the model outputs a 512-dimensional feature embedding for each face. However, enabling the `.classify = True` flag appends a classification head, allowing the model to directly output class logits over the **8,631 identities** present in the VGGFace2 training set.

In [ ]:
from facenet_pytorch import InceptionResnetV1

# ——————————————————————————————————————————————————————
#   Initialize the pre-trained face-recognition model
# ——————————————————————————————————————————————————————

# We use the InceptionResnetV1 architecture from the facenet-pytorch package,
# pre-trained on the VGGFace2 dataset for high-quality face embeddings.
# By calling .eval(), we set the model to inference mode (disables dropout, batchnorm updates).
# We then move the model to the appropriate device (GPU if available, else CPU).
nn1 = InceptionResnetV1(
    pretrained='vggface2'  # load weights trained on the VGGFace2 face dataset
).eval().to(device)         # switch to evaluation mode and transfer to GPU/CPU

# —————————————————————————————————————————————
#           Enable classification head
# —————————————————————————————————————————————

# By default, InceptionResnetV1 returns 512-dimensional embeddings.
# Setting .classify instructs the model to append a linear classification
# layer on top of the embeddings, so that nn1(input) returns raw class logits
# for all identities in VGGFace2 (8 631 classes), instead of embeddings.
nn1.classify = True

#### Face Detection and Alignment (MTCNN)

To ensure consistent and high-quality input for face recognition, the **MTCNN** (Multi-task Cascaded Convolutional Networks) model is used as a preprocessing stage. MTCNN carries out multiple tasks in sequence to detect and align faces with a high degree of accuracy. Specifically, it:

- Locates the most prominent face within each image.
- Aligns facial features based on detected landmarks (e.g., eyes, nose, and mouth).
- Crops and resizes the face region to a fixed resolution of **160×160 pixels**, matching the expected input size of the recognition model.

The aligned faces are returned as PyTorch tensors, ready for immediate use in model inference. Additionally, these preprocessed images can be optionally saved to disk, allowing the system to bypass real-time face detection in subsequent runs — a significant benefit in terms of efficiency.

> When the flag `mtcnn_processing_nn1` is set to `True`, the aligned face crops are automatically stored in the directory:  
> `testset/cropped_faces_nn1/{class_name}/img_XXX.jpg`

In [ ]:
# ———————————————————————————————————————————————————
#  Initialize the face detector and aligner (MTCNN)
# ———————————————————————————————————————————————————
# We use MTCNN from facenet-pytorch to detect, crop, and align faces in one step.
# When you call face_detector_nn1(img_batch), it returns a tensor of shape [B, 3, image_size, image_size]
# containing the aligned face crops, ready to feed into nn1 or an adversarial attack.

face_detector_nn1 = MTCNN(
    image_size=160,                             # int: output height/width of each face crop (default=160)
    margin=0,                                   # int: number of pixels to expand the face bounding box (default=0)
    min_face_size=20,                           # int: minimum face size (in pixels) that the detector will attempt to locate (default=20)
    thresholds=[0.6, 0.7, 0.7],                 # list of 3 floats: score thresholds for each detection stage—
                                                #   P-Net, R-Net, and O-Net respectively (default=[0.6, 0.7, 0.7])
    factor=0.709,                               # float: scale factor between pyramid levels; controls the search granularity (default=0.709)
    post_process=True,                          # bool: whether to apply face alignment post-processing (True)
    select_largest=True,                        # bool: if multiple faces are detected, return only the largest one (True)
    selection_method="center_weighted_size",    # str: heuristic for choosing among multiple detections—
                                                #   options include "largest" or "center_weighted_size" (default="center_weighted_size")
    keep_all=False,                             # bool: if True, return all detected faces; if False, return only one (default=False)
    device=device                               # torch.device or str: computation device, e.g. "cuda:0" or "cpu"
)


In this section, we prepare the test set so it can be efficiently used during inference. The aim is to create a `DataLoader` that iterates over face images, optionally applying preprocessing steps, and associates each image with the correct identity label.

The process includes the following components:

- **Image Transformations**  
  A basic image preprocessing pipeline is defined using `torchvision.transforms`. If MTCNN alignment is not enabled, each image is resized to 160×160 pixels (the input size required by the face recognition model) and then converted to a tensor.

- **Dataset Definition via ImageFolder**  
  The test images are loaded using `ImageFolder`, which expects the directory structure to be organised such that each identity has its own folder. The dataset automatically assigns a numerical label to each subfolder and loads all images accordingly.

- **Label Mapping**  
  A custom mapping `idx_to_class` is constructed to associate the internal numeric labels used by `ImageFolder` with the actual identity names provided in the test set metadata. This ensures label consistency throughout the evaluation process.

- **Final DataLoader**  
  The dataset is wrapped in a `DataLoader` for iteration. No parallel workers (`num_workers=0`) are used to maintain compatibility and simplicity. This `DataLoader` can now be used to either:
  - Apply face detection and alignment via MTCNN (if enabled), or
  - Pass pre-aligned images directly to the recognition model.

This structure ensures that the test set is handled in a clean, modular, and reproducible way.

In [ ]:
# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing_nn1 else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                  

The aligned dataset, which was stored in a PyTorch `DataLoader`, includes both the face images and their corresponding identity labels. Once loaded, the individual samples are unpacked and converted into two tensors: one containing all the aligned images, and the other containing the associated class labels.

To ensure compatibility with downstream processing steps, the image tensor is converted from PyTorch format to a NumPy array. This is especially useful when the data needs to be passed into frameworks such as **ART (Adversarial Robustness Toolbox)**, which often expect NumPy inputs for generating adversarial examples.

Finally, the script reconstructs a reverse label mapping that allows us to convert class names (i.e., the folder names used in `ImageFolder`) back into numerical indices. This is essential for consistent evaluation and interpretation of results, especially when comparing predicted and true labels.


In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset\\dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

# Inverti il dizionario per cercare per valore
class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

## 2. Generating Adversarial Examples

To conduct adversarial robustness experiments on the baseline face recognition model (NN1), the model is wrapped using the `PyTorchClassifier` interface provided by the Adversarial Robustness Toolbox (ART). This wrapper enables compatibility with ART's attack and defence modules, while preserving the model’s original behaviour.

The wrapping step exposes the structure of the model in a format that supports adversarial attacks, transforming it from a standard `PyTorch` module into an evaluation-ready artefact. The model is used in classification mode, with logits returned for all 8,631 identities present in the `VGGFace2` dataset. This is essential for both error-generic and error-specific attacks, which rely on direct manipulation of class predictions.

The input shape is fixed at `(3, 160, 160)`, matching the resolution of aligned face crops generated during preprocessing. Inputs are assumed to be normalised in the range `[-1, 1]`, and this constraint is enforced by the wrapper to prevent invalid perturbations.

Cross-entropy loss is used, which is standard for multi-class classification. Although no training is performed, an Adam optimiser with a learning rate of 0.01 is defined, as required by ART. This ensures compatibility with any future use cases involving fine-tuning or adversarial training.

The device is selected automatically (GPU if available), allowing efficient computation during attack generation. Once initialised, the resulting classifier object (`classifier_nn1`) is used as the core interface for generating adversarial examples, evaluating their effect, and producing security evaluation curves.

In [ ]:
import torch.nn as nn
import torch.optim as optim

classifier_nn1 = PyTorchClassifier(
    model=nn1,                                                     # The PyTorch model to use
    clip_values=(-1, 1),                                           # The minimum and maximum values of the input
    loss=nn.CrossEntropyLoss(),                                    # The loss function
    optimizer=optim.Adam(nn1.parameters(), lr=0.01),               # The optimizer
    input_shape=(3, 160, 160),                                     # The shape of the input
    nb_classes=LABELS.size,                                        # The number of classes
    device_type='cuda' if torch.cuda.is_available() else 'cpu'
)

attack_folder = os.path.join(results_folder, 'attack_results_nn1')

### FGSM (Fast Gradient Sign Method) Adversarial Attack

In preparation for evaluating the robustness of the NN1 model against adversarial examples, this section configures the necessary environment for applying the `Fast Gradient Sign Method (FGSM)`. The setup begins by importing the relevant attack class from the `Adversarial Robustness Toolbox` and setting a control flag (fsgm_generate) to determine whether the adversarial examples should be generated anew or retrieved from existing files.

To maintain a clear separation of outputs, the script systematically creates a series of directories. One directory is designated to store the adversarial images produced by FGSM, while another is reserved for evaluation results. Within the results directory, two subfolders are created to distinguish between the generic and specific error modes. The `“error generic”` folder will contain results assessing the model's overall vulnerability to untargeted perturbations, whereas the `“error specific”` folder will be used in the context of targeted attacks, where the goal is to mislead the model towards a particular incorrect identity.

A flag named `evaluate_all_targets` is also introduced. When enabled, it allows the evaluation of FGSM in a targeted fashion across all classes in the dataset. In this configuration, however, it is set to False to limit the scope of computation to a predefined target.

In [ ]:
from art.attacks.evasion import FastGradientMethod

fsgm_generate = True

fgsm_adv_folder = os.path.join(adversarial_folder, 'FGSM_nn1')
os.makedirs(fgsm_adv_folder, exist_ok=True)
fgsm_results_folder = os.path.join(attack_folder, 'FGSM_results')
os.makedirs(fgsm_results_folder, exist_ok=True)

fgsm_error_gen_folder = os.path.join(fgsm_results_folder, "FGSM_Error_Generic")
os.makedirs(fgsm_error_gen_folder, exist_ok=True)
fgsm_error_spec_folder = os.path.join(fgsm_results_folder, "FGSM_Error_Specific")
os.makedirs(fgsm_error_spec_folder, exist_ok=True)

evaluate_all_targets = False

fgsm_img_path_g = os.path.join(fgsm_error_gen_folder, "fgsm_error_generic_plot.png")
fgsm_sec_img_g = os.path.join(fgsm_error_gen_folder, "fgsm_error_generic_sec_plot.png")
fgsm_img_path_s = os.path.join(fgsm_error_spec_folder, "fgsm_error_specific_plot.png")
fgsm_sec_img_s = os.path.join(fgsm_error_spec_folder, "fgsm_error_specific_sec_plot.png")

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

This block performs the actual generation of adversarial examples using the **FGSM (Fast Gradient Sign Method)** in its `error-generic` configuration. The attack perturbs the input images by adding a small amount of noise in the direction of the gradient, aiming to push the model towards any incorrect prediction.

An epsilon value of `0.1` is used, representing the maximum perturbation applied to each pixel (scaled to the [-1, 1] input range). The attack is applied to the entire aligned test set.

Once the adversarial samples are generated, the model (`classifier_nn1`) is used to predict the new (potentially incorrect) labels. Both the adversarial inputs and the corresponding predictions are saved to disk for further evaluation and visualisation. This allows subsequent steps (such as metric computation and plotting) to be repeated without regenerating the samples.

In [ ]:
# Parametri
fgsm_epsilon = 0.1

if fsgm_generate:
    
    # Genera adversarial examples
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=fgsm_epsilon)
    x_test_adv_fgsm_g = fgsm.generate(x=x_test_aligned_nn1)

    # Predizioni
    y_test_adv_fgsm_g = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_fgsm_g), axis=1)] # Plot the target image
    
    # Ensure the test-set folder and the `samples/` subfolder exist
    if not os.path.exists(fgsm_adv_folder):
        os.makedirs(fgsm_adv_folder)
    # Save the adversarial examples and labels
    torch.save(x_test_adv_fgsm_g, fgsm_adv_folder + '/x_test_adv_fgsm_g.pt')
    torch.save(y_test_adv_fgsm_g, fgsm_adv_folder + '/y_test_adv_fgsm_g.pt')

Following the generation of adversarial examples, this section evaluates the NN1 classifier's performance under attack. The previously saved adversarial samples and their predicted labels are loaded from disk, along with the ground-truth labels for the clean test set. The evaluation is conducted using a standard accuracy metric, which quantifies the proportion of correctly classified adversarial inputs relative to the original labels.

A custom utility function `print_basic_metrics` is used to print key performance statistics such as the number of correct and incorrect predictions, and the maximum perturbation introduced by FGSM. The classification accuracy under attack is then computed using `accuracy_score`, providing a quantitative measure of the model's robustness at the chosen epsilon value. Finally, the function `plot_predicted_images` is invoked to visually compare the clean and adversarial images for a selected set of identities. This visualisation highlights whether the perturbations have altered the model's predictions and illustrates the nature of the misclassifications introduced by the attack.

In [ ]:
# Percorsi salvataggio
x_test_adv_fgsm_g = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_g.pt')
y_test_adv_fgsm_g = torch.load(fgsm_adv_folder + '/y_test_adv_fgsm_g.pt')

fgsm_epsilon = 0.1

print(f"=== FGSM Evaluation on NN1 (Error Generic) ===")
print(f"ε = {fgsm_epsilon}\n")
correct, incorrect = print_basic_metrics(y_true, y_test_adv_fgsm_g, x_adv=x_test_adv_fgsm_g, x_orig=x_test_aligned_nn1)
acc = accuracy_score(y_true, y_test_adv_fgsm_g)
print(f"Accuracy           : {acc*100:.2f}%")

plot_predicted_images(
    x_test         = x_test_aligned_nn1,           # numpy array o torch.Tensor
    x_adv          = x_test_adv_fgsm_g,               # numpy array o torch.Tensor   
    y_true         = y_true,                   # ground‐truth “pulite”
    y_pred         = y_test_adv_fgsm_g,               # predizioni sugli adversarial
    y_pred_adv     = y_test_adv_fgsm_g,               # predizioni sugli adversarial
    title          = f"FGSM (Error Generic) attack - (ε={fgsm_epsilon})",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = fgsm_img_path_g
)

Following the evalutation of the model with the adversarial examples, this section evaluates the NN1 classifier's robustness by constructing a `Security Evaluation Curve` (`SEC`) under untargeted FGSM attacks. The aim is to investigate how the model's predictive accuracy degrades in response to increasing adversarial perturbation (epsilon).

A predefined set of epsilon values is used to simulate varying levels of attack intensity. For each epsilon, a new batch of adversarial samples is generated using the Fast Gradient Sign Method. These adversarial inputs are then classified by the NN1 model, and the predictions are compared to the original ground-truth labels from the clean test set. The resulting classification accuracy is recorded for each level of perturbation.

To complement this quantitative analysis, the `print_basic_metrics` utility function is employed to summarise the model’s performance, reporting both the number of correct and incorrect classifications and the extent of perturbation introduced at each epsilon level. Once all evaluations are complete, the `plot_multiple_sec_curves` function is used to visualise the relationship between epsilon and accuracy, thereby offering a clear depiction of the model’s vulnerability as adversarial strength increases.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Generic)
# ──────────────────────────────────────────────────────────────────────────────

# Valori di epsilon da testare
epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
accuracies = [0.98]

print("=== Security Evaluation Curve: FGSM (Error Generic) ===\n")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps)
    x_adv = fgsm.generate(x=x_test_aligned_nn1)

    y_pred_adv = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_adv)
    accuracies.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
epsilons = [0] + epsilons
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (epsilons, accuracies, 'o', 'crimson', 'NN1')
    ],
    title="FGSM - Security Evaluation Curve (Error Generic)",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

plt.tight_layout()
plt.savefig(fgsm_sec_img_g, bbox_inches='tight')
plt.show()

#### Error Specific

Unlike the Error Generic setting, where the adversarial goal is simply to cause any misclassification, the `Error Specific variant of FGSM` aims to mislead the model into predicting a chosen target identity. In this case, the attack is repeated across all individuals in the test set to determine how easily each one can be impersonated through targeted perturbations.

If the `evaluate_all_targets` flag is enabled, the code iterates through every identity in the test set. For each target, a one-hot encoded label is constructed and used to guide a targeted FGSM attack with fixed epsilon. The attack attempts to modify all input images so that they are misclassified as the selected target.

After generating adversarial examples and predicting their labels using the NN1 classifier, the code computes the success rate—i.e., the proportion of inputs that the model misclassifies as the intended target. These results are collected and sorted, highlighting which identities are most vulnerable to being mimicked.

In [ ]:
if evaluate_all_targets:
    results = []
    eps = 0.1

    # Itera su ogni target nel test set
    for idx, row in test_set.iterrows():
        target_name = row["Name"]
        target_class = np.where(LABELS == target_name)[0]

        if len(target_class) == 0:
            print(f"[SKIP] Target {target_name} non trovato nei LABELS.")
            continue

        one_hot_target = to_categorical([target_class[0]], nb_classes=len(LABELS))[0]
        one_hot_targeted_label = np.tile(one_hot_target, (len(x_test_aligned_nn1), 1))

        attack = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=True)
        x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
        
        y_test_adv_fgsm_s = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
        
        success_rate = (np.array(y_test_adv_fgsm_s) == target_name).mean()
        results.append((target_name, success_rate))

        #print(f"[{idx+1:03}] Target = {target_name:<30} → Success Rate: {success_rate:.2%}")

    # Ordina i risultati
    results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

    # Mostra i migliori 10 target
    print("\nTop 10 target per targeted FGSM (eps = 0.1):")
    for name, sr in results_sorted[:10]:
        print(f"→ {name:<30} : {sr:.2%}")

The following section demonstrates a `targeted FGSM attack` where the adversarial goal is to have the model misclassify all test images as a specific identity — `Dave_Mustaine`. Unlike the error generic setting, where any misclassification is sufficient, the error specific variant imposes a stricter condition: the model must consistently predict the attacker-defined target class.

To begin, we use the helper function `generate_one_hot_target_and_plot_image` to:

- Create the one-hot encoding for the selected target class.

- Display a clean reference image of the target, providing a visual anchor for evaluating the effectiveness of the attack.

The attack itself is then executed using `FastGradientMethod` in `targeted mode`, with an `epsilon value of 0.1`. The one-hot vector is applied across the entire test set to guide the attack towards the specified class.

After generating the adversarial examples, predictions are made using the model, and the predicted labels are decoded for clarity. Both the adversarial inputs and their predicted labels are saved to disk for later evaluation or visual inspection.

In [ ]:
# Name of the target class
target_name = 'Dave_Mustaine'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

In [ ]:
fgsm_epsilon = 0.1

if fsgm_generate:

    attack = FastGradientMethod(estimator=classifier_nn1, eps=fgsm_epsilon, targeted=True)

    x_test_adv_fgsm_s = attack.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    y_test_adv_fgsm_s = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_fgsm_s), axis=1)]
    
    if not os.path.exists(fgsm_adv_folder):
        os.makedirs(fgsm_adv_folder)
    
    # Save the adversarial examples and labels
    torch.save(x_test_adv_fgsm_s, fgsm_adv_folder + '/x_test_adv_fgsm_s.pt')
    torch.save(y_test_adv_fgsm_s, fgsm_adv_folder +'/y_test_adv_fgsm_s.pt')

After generating adversarial examples for the targeted attack, this section evaluates their impact on the model's predictions. The adversarial inputs and their predicted labels are first loaded from disk.

The function `print_basic_metrics` is used to summarise the classification performance, reporting how many predictions remained correct and how many were altered under adversarial conditions. Additionally, the standard classification accuracy is computed to quantify the overall drop in model performance.

Most importantly in the context of a targeted attack, the `targeted success rate` is calculated. This metric indicates the percentage of adversarial inputs that the model misclassified as the specific target class — in this case, `Dave_Mustaine`. A high success rate reflects the model’s susceptibility to being manipulated toward this identity.

To complement the quantitative analysis, `the plot_predicted_images` function is used to visually compare clean and adversarial images for a subset of individuals. 

In [ ]:
# —————————————————————————————————————————————————————————————
# Error Specific FGSM (Targeted)
# —————————————————————————————————————————————————————————————
x_test_adv_fgsm_s = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_s.pt')
y_test_adv_fgsm_s = torch.load(fgsm_adv_folder + '/y_test_adv_fgsm_s.pt')

# compute and print metrics
corr, incorr = print_basic_metrics(y_true, y_test_adv_fgsm_s, x_orig=x_test_aligned_nn1, x_adv =x_test_adv_fgsm_s)
acc = accuracy_score(y_true, y_test_adv_fgsm_s)

sr  = (np.array(y_test_adv_fgsm_s) == target_name).mean() * 100

fgsm_epsilon = 0.1

print(f"=== FGSM Error Specific ===")
print(f"ε = {fgsm_epsilon}, target = {target_name}")
print(f"Accuracy         : {acc*100:.2f}%")
print(f"Targeted success : {sr:.2f}%\n")

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_fgsm_s,
    y_true         = y_true,
    y_pred         = y_test_adv_fgsm_s,
    y_pred_adv     = y_test_adv_fgsm_s,
    title          = f"FGSM Error Specific - ε={fgsm_epsilon}, target {target_name}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = fgsm_img_path_s
)

This analysis mirrors the one previously conducted in the Error Generic setting, where adversarial examples were generated using FGSM across increasing epsilon values to observe the degradation in classification accuracy. However, in this case, the attack is performed in Error Specific mode, meaning the perturbations are crafted to force predictions toward a fixed target identity.

For each epsilon, two metrics are computed. The first is the targeted success rate, which measures how frequently the model predicts the selected identity (`Dave_Mustaine`) as intended by the attack. The second is the overall accuracy, capturing the model’s performance across all classes under adversarial conditions.

The structure of the code and the evaluation process remain largely the same as in the generic case, but the targeted setting introduces an additional perspective on the model's behaviour: rather than just failing, the model is being deliberately steered toward a specific output. The results are visualised using `plot_multiple_sec_curves`, where both the targeted success rate and general accuracy are plotted against epsilon. 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Specific)
# ──────────────────────────────────────────────────────────────────────────────

epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]
success_rates = [0.01]
accuracies = [0.98]

print("=== Security Evaluation Curve: FGSM (Error Specific vs General Accuracy) ===\n")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=True)
    x_adv = fgsm.generate(x_test_aligned_nn1, one_hot_targeted_label)
    y_pred_adv = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    
    sr = (np.array(y_pred_adv) == target_name).mean()
    success_rates.append(sr)
    print(f"   Success Rate: {sr*100:.2f}%")
    
    acc = accuracy_score(y_true, y_pred_adv)
    accuracies.append(acc)
    print(f"   Accuracy All:  {acc*100:.2f}%")

    _, _ = print_basic_metrics(y_true, y_pred_adv, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
epsilons = [0] + epsilons
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (epsilons, success_rates, 'o', 'blue', 'Success Rate (Targeted)'),
        (epsilons, accuracies, 's', 'crimson', 'Accuracy (All Classes)')
    ],
    title="FGSM - Security Evaluation Curve (Targeted vs All)",
    xlabel="Epsilon",
    ylabel="Metric"
)

plt.tight_layout()
plt.savefig(fgsm_sec_img_s, bbox_inches='tight')
plt.show()

### BIM (Basic Iterative Method) Adversarial Attack 

Continuing the analysis of the model’s robustness, we now turn to the evaluation of adversarial examples generated using the `Basic Iterative Method` (`BIM`). Unlike FGSM, which applies a single-step perturbation, BIM introduces the adversarial noise iteratively over several small updates, allowing it to more effectively exploit model vulnerabilities.

To support this evaluation, the necessary folder structure is set up. A dedicated directory is created to store the adversarial images generated by BIM, and a separate results folder is organised to contain evaluation outputs. As in previous configurations, two subfolders are defined to distinguish between the error generic and error specific modes. The first one will hold results where the objective is general misclassification, while the second ones will be used when attempting to force the model to predict a specific, predefined identity.

In [ ]:
from art.attacks.evasion import BasicIterativeMethod

bim_generate = True

bim_adv_folder = os.path.join(adversarial_folder, 'BIM_nn1')
os.makedirs(bim_adv_folder, exist_ok=True)
bim_results_folder = os.path.join(attack_folder, 'BIM_results')
os.makedirs(bim_results_folder, exist_ok=True)

bim_error_gen_folder = os.path.join(bim_results_folder, "BIM_Error_Generic")
os.makedirs(bim_error_gen_folder, exist_ok=True)
bim_error_spec_folder = os.path.join(bim_results_folder, "BIM_Error_Specific")
os.makedirs(bim_error_spec_folder, exist_ok=True)

bim_img_path_g = os.path.join(bim_error_gen_folder, "bim_error_generic_plot.png")
bim_sec_img_g = os.path.join(bim_error_gen_folder, "bim_error_generic_sec_plot.png")
bim_img_path_s = os.path.join(bim_error_spec_folder, "bim_error_specific_plot.png")
bim_sec_img_s = os.path.join(bim_error_spec_folder, "bim_error_specific_sec_plot.png")

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

This block handles the generation of adversarial examples using the Basic Iterative Method (BIM) under the `error-generic configuration`. BIM extends the idea of FGSM by applying multiple small perturbation steps, rather than a single gradient update, allowing it to more effectively deceive the model.

The attack is configured with an `epsilon value of 0.03`, which defines the maximum total perturbation applied to each pixel. Each step of the attack applies a `change of 0.01`, and the process is repeated for up to `3 iterations`. These parameters ensure that the adversarial noise remains bounded while giving the attack enough flexibility to steer the input towards misclassification.

The perturbed samples are generated from the aligned test set and passed through the classifier (`classifier_nn1`) to obtain the new predicted labels. Both the adversarial examples and their corresponding predictions are saved to disk.

In [ ]:
bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

if bim_generate:
    
    # Genera adversarial examples
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=bim_eps,
        eps_step=bim_eps_step,
        max_iter=bim_max_iter
    )
    
    x_test_adv_bim_g = bim.generate(x=x_test_aligned_nn1)

    # Predizioni
    y_test_adv_bim_g = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_bim_g),axis=1)] # Plot the target image
    
    # Ensure the test-set folder and the `samples/` subfolder exist
    if not os.path.exists(bim_adv_folder):
        os.makedirs(bim_adv_folder)
        
    # Save the adversarial examples and labels
    torch.save(x_test_adv_bim_g, bim_adv_folder + '/x_test_adv_bim_g.pt')
    torch.save(y_test_adv_bim_g, bim_adv_folder + '/y_test_adv_bim_g.pt')

Following the generation of adversarial samples using the Basic Iterative Method, this block evaluates their effect on the classifier’s performance in the error-generic setting — where the goal is simply to cause any misclassification, without targeting a specific identity.

The previously saved adversarial examples and their predicted labels are loaded from disk. The overall accuracy is computed to quantify how much the attack has degraded the model’s performance under the specified parameters: a `maximum perturbation` (`ε`) of `0.03`, a `step size` of `0.01`, and a limit of `3 iterations`.

In addition to the accuracy score, the function `print_basic_metrics` is called to display a breakdown of correct versus incorrect predictions, as well as the maximum perturbation magnitude. This provides an immediate sense of the attack’s effectiveness and perceptual subtlety.

Finally, `plot_predicted_images` is used to visualise a sample of adversarial predictions in comparison to the clean inputs.

In [ ]:
# —————————————————————————————————————————————————————————————
# BIM (Basic Iterative Method) - Error Generic
# —————————————————————————————————————————————————————————————

x_test_adv_bim_g = torch.load(bim_adv_folder + '/x_test_adv_bim_g.pt')
y_test_adv_bim_g = torch.load(bim_adv_folder + '/y_test_adv_bim_g.pt')

acc_bim = accuracy_score(y_true, y_test_adv_bim_g)

bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

print("=== BIM Evaluation (Error Generic) ===")
print(f"ε = {bim_eps}, ε_step = {bim_eps_step}, iter = {bim_max_iter}")
print(f"Accuracy           : {acc_bim*100:.2f}%\n")
correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_bim_g,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_bim_g
)

plot_predicted_images(
    x_test         = x_test_adv_bim_g,
    x_adv          = x_test_adv_bim_g,
    y_true         = y_true,
    y_pred         = y_test_adv_bim_g,
    y_pred_adv     = y_test_adv_bim_g,
    title          = f"BIM (Error Generic) - ε={bim_eps}, step={bim_eps_step}, iter={bim_max_iter}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = bim_img_path_g
)

As done previously with FGSM, this section analyses the model’s robustness to adversarial attacks using `Security Evaluation Curves`, this time applied to the Basic Iterative Method (BIM) under the error generic setting. The evaluation focuses on how the classifier’s accuracy is affected by varying the three main parameters of the BIM attack.

Three different experiments are conducted, each exploring the impact of a specific attack parameter while keeping the others fixed:

- **Varying `ε`**: The maximum total perturbation allowed is changed across a defined range, while keeping ε_step and the number of iterations constant. This shows how sensitive the model is to stronger perturbations.

- **Varying `ε_step`**: The step size of each iteration is varied, with fixed ε and iteration count. This helps assess whether finer or coarser steps make the attack more effective.

- **Varying `max_iter`**: The number of iterations is adjusted while keeping both ε and ε_step constant. This tests how the depth of the iterative attack influences the outcome.

For each configuration, adversarial examples are generated and evaluated against the clean ground-truth labels. Accuracy is computed and plotted using `plot_multiple_sec_curves`, producing three comparative graphs. These plots help visualise the trade-off between attack strength and model accuracy, and they offer insight into which BIM parameters most significantly impact the model's robustness.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Generic)
# —————————————————————————————————————————————————————————————

eps_values       = [0.01, 0.03, 0.05, 0.1]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]

eps              = 0.03
eps_step         = 0.01
max_iter         = 3

curve_colors = ['crimson', 'darkorange', 'seagreen']

# 1) Accuracy vs ε (fixed eps_step and max_iter)
accuracies_eps  = [0.98]

print("=== SEC: BIM vs ε ===")
for epsilon in eps_values:
    print(f"\n→ Generating adversarial examples eps = {epsilon:.3f}, eps_step = {eps_step:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=epsilon,
        eps_step=eps_step,
        max_iter=max_iter
    )
    x_adv      = bim.generate(x=x_test_aligned_nn1)
    y_pred_adv = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc        = accuracy_score(y_true, y_pred_adv)
    accuracies_eps.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, accuracies_eps, 'o', curve_colors[0], 'NN1')
    ],
    title=f"Accuracy vs ε with (ε_step={eps_step}, iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# 2) Accuracy vs ε_step (fixed eps and max_iter)
accuracies_step  = [0.98]

print("\n=== SEC: BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {eps:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=step,
        max_iter=max_iter
    )
    x_adv      = bim.generate(x=x_test_aligned_nn1)
    y_pred_adv = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc        = accuracy_score(y_true, y_pred_adv)
    accuracies_step.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, accuracies_step, 's', curve_colors[1], 'NN1')
    ],
    title=f"Accuracy vs Epsilon Step with (ε={eps}, iter={max_iter})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)


# 3) Accuracy vs max_iter (fixed eps and eps_step)
accuracies_iter   = [0.98]

print("\n=== SEC: BIM vs max_iter ===")
for n_iter in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {n_iter}, eps = {eps:.3f}, eps_step = {eps_step:.3f}")
    bim = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=eps_step,
        max_iter=n_iter
    )
    x_adv      = bim.generate(x=x_test_aligned_nn1)
    y_pred_adv = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc        = accuracy_score(y_true, y_pred_adv)
    accuracies_iter.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, accuracies_iter, '^', curve_colors[2], 'NN1')
    ],
    title=f"Accuracy vs Max Iterations with (ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig(bim_sec_img_g, bbox_inches='tight')
plt.show()

#### Error Specific

Continuing the robustness evaluation in the `error specific setting`, this block configures a `targeted adversarial attack using the Basic Iterative Method (BIM)`. The objective is to force the model to misclassify all test samples as a single identity — in this case, a predefined target class `Fernando_Torres`.

To start, the `generate_one_hot_target_and_plot_image` utility is used to both construct the required one-hot label for the chosen identity and to display a clean reference image of that identity for visual context.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

The attack is launched using BIM in targeted mode, with the following parameters:

- Maximum perturbation `ε = 0.03`

- Step size `ε_step = 0.01`

- Iterations `max_iter = 3`

These settings guide the model to iteratively adjust each input so that it is misclassified as the target identity, staying within the allowed perturbation bounds.

After generating the adversarial inputs, predictions are made using the original classifier. The adversarial samples and their corresponding predicted labels are then saved to disk for future inspection or evaluation, following the same logic used in the FGSM targeted case — but now taking advantage of BIM’s iterative refinement for potentially stronger attacks.

In [ ]:
bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

if bim_generate:

    attack = BasicIterativeMethod(
        estimator=classifier_nn1,
        eps=bim_eps,
        eps_step=bim_eps_step,
        max_iter=bim_max_iter,
        targeted=True
    )

    x_test_adv_bim_s = attack.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    y_test_adv_bim_s = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_bim_s), axis=1)] # Plot the target image
    
    if not os.path.exists(bim_adv_folder):
        os.makedirs(bim_adv_folder)
    
    torch.save(x_test_adv_bim_s, bim_adv_folder + '/x_test_adv_bim_s.pt')
    torch.save(y_test_adv_bim_s, bim_adv_folder +'/y_test_adv_bim_s.pt')

Same as before, this section evaluates the effectiveness of the Basic Iterative Method (BIM) when used to carry out a targeted adversarial attack. The objective remains to coerce the model into consistently predicting a specific identity—Fernando_Torres—regardless of the input image.

The adversarial examples and their corresponding predictions, previously saved to disk, are loaded and compared against the clean ground-truth labels. Two key metrics are computed: the overall classification accuracy under attack and the targeted success rate, which measures how often the model misclassifies inputs as the chosen target class.

The `print_basic_metrics` function provides a summary of prediction correctness and perturbation magnitude, helping quantify the attack's impact. In parallel, `plot_predicted_images` generates a visual comparison between clean and adversarial samples, illustrating the extent to which the perturbations have altered the model’s behaviour.

In [ ]:
# —————————————————————————————————————————————————————————————
# BIM Error Specific (Targeted)
# —————————————————————————————————————————————————————————————

x_test_adv_bim_s = torch.load(bim_adv_folder + '/x_test_adv_bim_s.pt')
y_test_adv_bim_s = torch.load(bim_adv_folder + '/y_test_adv_bim_s.pt')

acc = accuracy_score(y_true, y_test_adv_bim_s)
sr  = (np.array(y_test_adv_bim_s) == target_name).mean() * 100

bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

print(f"=== BIM Error Specific (Targeted) ===")
print(f"ε = {bim_eps}, ε_step = {bim_eps_step}, iter = {bim_max_iter}")
print(f"Target class        : {target_name}")
print(f"Accuracy            : {acc*100:.2f}%")
print(f"Targeted success    : {sr:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_bim_s,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_bim_s
)

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_bim_s,
    y_true         = y_true,
    y_pred         = y_test_adv_bim_s,
    y_pred_adv     = y_test_adv_bim_s,
    title          = f"BIM Targeted - ε={bim_eps}, target {target_name}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = bim_img_path_s
)

Once again, as done previously for the FGSM attack, this section evaluates the model’s performance under the `BIM attack in the error specific (targeted)` setting, by varying its main hyperparameters.

The goal is to force the model to classify all test images as a predefined target identity, regardless of their true class. To assess the effectiveness of the attack, we construct `Security Evaluation Curves (SEC)` that measure both overall `accuracy` and `targeted success rate` under the following conditions:

Three sets of curves are plotted:

- **Accuracy and Targeted Accuracy vs `ε`**: Shows how both metrics evolve as the overall perturbation strength increases.

- **Accuracy and Targeted Accuracy vs `ε_step`**: Examines the impact of different step sizes while keeping ε and iteration count fixed.

- **Accuracy and Targeted Accuracy vs `max_iter`**: Evaluates how the number of iterations influences the model’s robustness, keeping ε and ε_step constant.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Specific - Targeted)
# —————————————————————————————————————————————————————————————

eps_values      = [0.01, 0.03, 0.05, 0.1]
eps_step_values = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values = [1, 3, 5, 10, 15]

eps             = 0.03
eps_step        = 0.01
max_iter        = 3

print("=== Security Evaluation Curves - BIM (Error Specific) ===")

curve_colors    = ['crimson', 'blue']

# 1) Accuracy & Targeted Success vs ε
acc_eps         = [0.98]
sr_eps          = [0.01]

print("=== SEC: BIM vs ε ===")
for epsilon in eps_values:
    print(f"\n→ Generating adversarial examples eps = {epsilon:.3f}, eps_step = {eps_step}, iter = {max_iter}")
    attack = BasicIterativeMethod(
        estimator   = classifier_nn1,
        eps         = epsilon,
        eps_step    = eps_step,
        max_iter    = max_iter,
        targeted    = True   
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Epsilon vs Accuracy / Targeted Accuracy\n(ε_step={eps_step}, iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# 2) Accuracy & Targeted Accuracy vs ε_step
acc_step        = [0.98]
sr_step         = [0.01]

print("\n=== SEC: BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {eps}, iter = {max_iter}")
    attack = BasicIterativeMethod(
        estimator   = classifier_nn1,
        eps         = eps,
        eps_step    = step,
        max_iter    = max_iter,
        targeted    = True   
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Epsilon Step vs Accuracy / Targeted Accuracy\n(ε={eps}, iter={max_iter})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)

# 3) Accuracy & Targeted Accuracy vs max_iter
acc_iter    = [0.98]
sr_iter     = [0.01]

print("\n=== SEC: BIM vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}, eps = {eps}, eps_step = {eps_step}")
    attack = BasicIterativeMethod(
        estimator   = classifier_nn1,
        eps         = eps,
        eps_step    = eps_step,
        max_iter    = it,
        targeted    = True   
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Max Iterations vs Accuracy / Targeted Accuracy\n(ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig(bim_sec_img_s, bbox_inches='tight')
plt.show()

### PGD (Projected Gradient Descent) Adversarial Attack

Now we focus on evaluating adversarial examples crafted using the `Projected Gradient Descent` (`PGD`) method. PGD is a widely used and powerful attack that extends BIM by adding a projection step to ensure that each perturbed input remains within a defined constraint set. This makes it especially effective at generating robust and bounded adversarial examples, often used as a benchmark in adversarial robustness research.

In [ ]:
from art.attacks.evasion import ProjectedGradientDescent

pgd_generate = True

pgd_adv_folder = os.path.join(adversarial_folder, 'PGD_nn1')
os.makedirs(pgd_adv_folder, exist_ok=True)
pgd_results_folder = os.path.join(attack_folder, 'PGD_results')
os.makedirs(pgd_results_folder, exist_ok=True)

pgd_error_gen_folder = os.path.join(pgd_results_folder, "PGD_Error_Generic")
os.makedirs(pgd_error_gen_folder, exist_ok=True)
pgd_error_spec_folder   = os.path.join(pgd_results_folder, "PGD_Error_Specific")
os.makedirs(pgd_error_spec_folder, exist_ok=True)

pgd_img_path_g = os.path.join(pgd_error_gen_folder, "pgd_error_generic_plot.png")
pgd_sec_img_g = os.path.join(pgd_error_gen_folder, "pgd_error_generic_sec_plot.png")
pgd_img_path_s  = os.path.join(pgd_error_spec_folder, "pgd_error_specific_plot.png")
pgd_sec_img_s = os.path.join(pgd_error_spec_folder, "pgd_error_specific_sec_plot.png")

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

The following block performs the generation of adversarial examples using the Projected Gradient Descent (PGD) method in the `error generic setting`. PGD builds upon the iterative approach used in BIM but introduces multiple random initialisations and a projection step to ensure that perturbations remain within a constrained norm-ball around the original input.

The attack is configured with a maximum perturbation `ε = 0.03`, a `step size` of `0.01`, and up to `3iterations`. In addition, `num_random_init = 5` specifies that the attack should be run with five different random initial conditions per input, increasing the chance of finding a successful adversarial example.

After generating the perturbed samples, the classifier is used to predict their labels, which are expected to differ from the originals in a successful attack. Both the adversarial examples and their corresponding predictions are saved to disk, ensuring that further evaluations and visual analyses can be performed without regenerating the data.

In [ ]:
# ————————————————————————————————————————————————————
# PGD Error Generic
# ————————————————————————————————————————————————————

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
num_random_init = 5

if pgd_generate:

    pgd = ProjectedGradientDescent(
        estimator        = classifier_nn1,
        eps              = pgd_eps,
        eps_step         = pgd_eps_step,
        max_iter         = pgd_max_iter,
        num_random_init  = num_random_init,
        targeted=False
    )

    x_test_adv_pgd_g = pgd.generate(x=x_test_aligned_nn1)

    y_test_adv_pgd_g = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_pgd_g), axis=1)]
    
    if not os.path.exists(pgd_adv_folder):
        os.makedirs(pgd_adv_folder)
        
    # Save the adversarial examples and labels
    torch.save(x_test_adv_pgd_g, pgd_adv_folder + '/x_test_adv_pgd_g.pt')
    torch.save(y_test_adv_pgd_g, pgd_adv_folder + '/y_test_adv_pgd_g.pt')

The previously generated adversarial examples and their predicted labels are loaded from disk.

The evaluation begins by computing the overall accuracy under attack conditions, using the specified PGD parameters specified previously to generate the adversarial samples. The `num_random_init` parameter allows the attack to start from multiple initial perturbations, increasing its effectiveness in escaping local minima and maximising misclassification.

In [ ]:
x_test_adv_pgd_g = torch.load(pgd_adv_folder + '/x_test_adv_pgd_g.pt')
y_test_adv_pgd_g = torch.load(pgd_adv_folder + '/y_test_adv_pgd_g.pt')

acc = accuracy_score(y_true, y_test_adv_pgd_g)

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
num_random_init = 5

print("=== PGD Evaluation (Error Generic) ===")
print(f"ε = {pgd_eps}, ε_step = {pgd_eps_step}, iter = {pgd_max_iter}, init = {num_random_init}")
print(f"Accuracy          : {acc*100:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_pgd_g,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_pgd_g
)
plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_pgd_g,
    y_true         = y_true,
    y_pred         = y_test_adv_pgd_g,
    y_pred_adv     = y_test_adv_pgd_g,
    title          = f"PGD Attack (Error Generic) - ε={pgd_eps}",
    id_test_images = idx_test_images,
    image_idx      = 0
)

So, we procede to evaluates the robustness of the model against Projected Gradient Descent (PGD) attacks by analysing how classification accuracy changes as the key attack parameters vary. The evaluation is conducted in the error generic setting.

Four distinct evaluation curves are produced, each exploring the effect of varying a specific attack parameter:

1. **Accuracy vs `ε`** — Tests how increasing the maximum perturbation impacts model performance.

2. **Accuracy vs `ε_step`** — Observes the effect of varying step sizes per iteration, while keeping total perturbation and iteration count fixed.

3. **Accuracy vs `max_iter`** — Assesses how the number of attack iterations affects the model’s ability to resist misclassification.

4. **Accuracy vs `num_random_init`** — Explores the impact of using multiple random initialisations, which can increase the strength and variability of the attack.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - PGD (Error Generic)
# —————————————————————————————————————————————————————————————


eps_values       = [0.01, 0.03, 0.05, 0.10]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]
num_init_values  = [1, 3, 5]

pgd_eps          = 0.03
pgd_eps_step     = 0.01
pgd_max_iter     = 3
num_random_init  = 5

curve_colors = ['crimson', 'darkorange', 'seagreen', 'royalblue']

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps = [0.98]

print("=== SEC: PGD vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples ε = {eps:.3f}")
    pgd = ProjectedGradientDescent(
        estimator       = classifier_nn1,
        eps             = eps,
        eps_step        = pgd_eps_step,
        max_iter        = pgd_max_iter,
        num_random_init = num_random_init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    acc_eps.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'NN1')
    ],
    title=f"Accuracy vs Epsilon (ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step = [0.98]

print("\n=== SEC: PGD vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples ε_step = {step:.3f}")
    pgd = ProjectedGradientDescent(
        estimator       = classifier_nn1,
        eps             = pgd_eps,
        eps_step        = step,
        max_iter        = pgd_max_iter,
        num_random_init = num_random_init,
        targeted        = False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    acc_step.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 's', curve_colors[1], 'NN1')
    ],
    title=f"Accuracy vs Epsilon Step (ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter = [0.98]

print("\n=== SEC: PGD vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}")
    pgd = ProjectedGradientDescent(
        estimator       = classifier_nn1,
        eps             = pgd_eps,
        eps_step        = pgd_eps_step,
        max_iter        = it,
        num_random_init = num_random_init,
        targeted        = False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    acc_iter.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, '^', curve_colors[2], 'NN1')
    ],
    title=f"Accuracy vs Max iterations (ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max iterations",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
acc_init = [0.98]

print("\n=== SEC: PGD vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples num_random_init = {init}")
    pgd = ProjectedGradientDescent(
        estimator       = classifier_nn1,
        eps             = pgd_eps,
        eps_step        = pgd_eps_step,
        max_iter        = pgd_max_iter,
        num_random_init = init,
        targeted        = False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    acc_init.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_nn1, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (num_init_values, acc_init, 'd', curve_colors[3], 'NN1')
    ],
    title=f"Accuracy vs Random Initializations (ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(pgd_sec_img_g, bbox_inches='tight')
plt.show()

#### Error Specific

Now we continue with the generation of targeted adversarial examples using **Projected Gradient Descent (PGD)** in the `error specific` setting. The objective is to craft perturbations that cause the model to consistently misclassify all inputs as a predefined identity — `Fernando_Torres` — regardless of the true label.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

The attack is configured with `ε = 0.03`, a step size of `0.01`, a maximum of `3` iterations, and `5` random initialisations, same as the error generic setting. These settings allow the attack to explore multiple starting points within the perturbation space, increasing the likelihood of successful targeted misclassification.

The target label is encoded as a one-hot vector and broadcast across the test set to ensure each input is directed toward the same identity. The PGD attack is executed in targeted mode, and the resulting adversarial predictions are computed using the classifier.

Both the adversarial inputs and their predicted labels are saved to disk to enable consistent evaluation and visual analysis in later stages.

In [ ]:
pgd_eps       = 0.03
pgd_eps_step  = 0.01
pgd_max_iter  = 3
pgd_init      = 5

if pgd_generate:
    
    # Genera adversarial (targeted)
    pgd = ProjectedGradientDescent(
        estimator        = classifier_nn1,
        eps              = pgd_eps,
        eps_step         = pgd_eps_step,
        max_iter         = pgd_max_iter,
        num_random_init  = pgd_init,
        targeted         = True        
    )

    x_test_adv_pgd_s = pgd.generate(x_test_aligned_nn1, one_hot_targeted_label)

    # predict on adversarials
    y_test_adv_pgd_s = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_pgd_s), axis=1)]

    # Ensure the test-set folder and the `samples/` subfolder exist
    if not os.path.exists(pgd_adv_folder):
        os.makedirs(pgd_adv_folder)
        
    # Save the adversarial examples and labels
    torch.save(x_test_adv_pgd_s, pgd_adv_folder + '/x_test_adv_pgd_s.pt')
    torch.save(y_test_adv_pgd_s, pgd_adv_folder +'/y_test_adv_pgd_s.pt')

The adversarial examples and their predicted labels are then loaded from disk, alongside the ground-truth labels for the clean test set. Two key metrics are computed: the overall classification accuracy, which reflects the model's ability to resist misclassification, and the targeted success rate, which quantifies how often the model predicts the intended target class.

In [ ]:
# ————————————————————————————————————————————————————
# PGD Error Specific (Targeted)
# ————————————————————————————————————————————————————

x_test_adv_pgd_s = torch.load(pgd_adv_folder + '/x_test_adv_pgd_s.pt')
y_test_adv_pgd_s = torch.load(pgd_adv_folder + '/y_test_adv_pgd_s.pt')

pgd_eps       = 0.03
pgd_eps_step  = 0.01
pgd_max_iter  = 3
pgd_init      = 5

acc   = accuracy_score(y_true, y_test_adv_pgd_s)
sr    = (np.array(y_test_adv_pgd_s) == target_name).mean() * 100

print("=== PGD Error Specific ===")
print(f"ε = {pgd_eps}, step = {pgd_eps_step}, iter = {pgd_max_iter}, init = {pgd_init}")
print(f"Target           : {target_name}")
print(f"Accuracy         : {acc*100:.2f}%")
print(f"Targeted success : {sr:.2f}%\n")
correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_pgd_s,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_pgd_s
)
plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_pgd_s,
    y_true         = y_true,
    y_pred         = y_test_adv_pgd_s,
    y_pred_adv     = y_test_adv_pgd_s,
    title          = f"PGD Error Specific - ε={pgd_eps}, target {target_name}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = pgd_img_path_s
)

The evaluation is then performed by varying attack parameters and observing their impact on both overall accuracy and targeted success rate.

Four experimental setups are explored:

1. **Accuracy and Targeted Accuracy vs `ε`** - This evaluates how increasing the total allowed perturbation influences the ability of the attack to cause misclassification and enforce the target label.

2. **Accuracy and Targeted Accuracy vs `ε_step`** - Here, the step size per iteration is varied to assess whether finer or coarser updates yield better attack performance, under a fixed total perturbation.

3. **Accuracy and Targeted Accuracy vs `max_iter`** - This investigates the effect of increasing the number of iterations, determining how attack depth affects both general misclassification and targeted success.

4. **Accuracy and Targeted Accuracy vs `num_random_init`** - This examines the impact of using multiple random starting points for the attack. More initialisations can improve the effectiveness of the attack by helping it escape poor local optima.

In [ ]:
# —————————————————————————————————————————————————————————————
# PGD Targeted - Security Evaluation Curves (Error Specific)
# —————————————————————————————————————————————————————————————

# Parameter space
eps_values      = [0.01, 0.03, 0.05, 0.10]
eps_step_values = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values = [1, 3, 5, 10, 15]
num_init_values = [1, 3, 5]

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 10
num_random_init = 5

curve_colors     = ['crimson', 'darkorange', 'seagreen', 'royalblue']

# —————————————————————————————————————————————————————————————
# 1) Accuracy & Targeted Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps, sr_eps = [0.98], [0.01]

print("=== SEC: PGD vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples ε = {eps:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon \n(ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy & Targeted Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step, sr_step = [0.98], [0.01]

print("\n=== SEC: PGD vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples ε_step = {step:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon Step \n(ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsion Step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy & Targeted Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter, sr_iter = [0.98], [0.01]

print("\n=== SEC: PGD vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=it,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Max Iterations\n(ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy & Targeted Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
acc_init, sr_init = [0.98], [0.01]

print("\n=== SEC: PGD vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples num_random_init = {init}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    y_pred_nn1 = LABELS[np.argmax(classifier_nn1.predict(x_adv), axis=1)]
    acc = accuracy_score(y_true, y_pred_nn1)
    sr  = (np.array(y_pred_nn1) == target_name).mean()
    acc_init.append(acc)
    sr_init.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (num_init_values, acc_init, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (num_init_values, sr_init,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Random Initializations\n(ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(pgd_sec_img_s, bbox_inches='tight')
plt.show()

### Deepfool - Adversarial Attack

We now turn to the evaluation of adversarial examples generated using the DeepFool attack. Unlike PGD or BIM, which rely on fixed perturbation thresholds and step-wise updates, `DeepFool` is an untargeted, iterative attack that dynamically computes the minimal perturbation required to push an input across the decision boundary of the classifier.

To prepare for this analysis, the code sets up the necessary directory structure. A folder is created to store adversarial examples produced by DeepFool, while a separate results directory is used for evaluation outputs. Within this, a subfolder is dedicated to the error generic setting, in which the goal is to cause any misclassification without targeting a specific class.

In [ ]:
from art.attacks.evasion import DeepFool

deepfool_generate = True

# Cartelle e percorsi
deepfool_adv_folder = os.path.join(adversarial_folder, "Deepfool_nn1")
os.makedirs(deepfool_adv_folder, exist_ok=True)
deepfool_folder   = os.path.join(attack_folder, "Deepfool_results")
os.makedirs(deepfool_folder, exist_ok=True)

deepfool_sec_folder = os.path.join(deepfool_folder, "Deepfool_Error_Generic")
os.makedirs(deepfool_sec_folder, exist_ok=True)

deepfool_img_path = os.path.join(deepfool_sec_folder, "deepfool_plot.png")
deepfool_img_path = os.path.join(deepfool_sec_folder, "deepfool_sec_plot.png")

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

We start again with the generation of adversarial examples using the DeepFool attack in the `error generic setting`, where the objective is to minimally perturb each input so that it crosses the classifier's decision boundary and is misclassified.

Unlike other methods such as FGSM or PGD, which apply fixed or iterative noise within a specified threshold, DeepFool dynamically computes the smallest possible perturbation required to change the model’s prediction. It operates under an ℓ₂ norm constraint and refines the perturbation through a series of linear approximations to the classifier’s decision boundaries.

The attack is configured with a convergence threshold `ε = 1e-4` and a maximum of `1 iterations` per input. Due to its sample-wise nature, the attack is applied individually to each image in the test set. For each input, the resulting adversarial example is collected, and the predicted label is stored.

Once the entire set has been processed, the adversarial examples and their corresponding predictions are stacked and saved to disk.

In [ ]:
deepfool_epsilon      = 1e-4
deepfool_max_iter     = 1

if deepfool_generate:
    
    deepfool = DeepFool(classifier=classifier_nn1, epsilon=deepfool_epsilon, max_iter=deepfool_max_iter, verbose=False)

    x_test_adv_deepfool = []
    y_test_adv_deepfool = []
    
    for i, x_orig in enumerate(tqdm(x_test_aligned_nn1, leave=False)):
        x_input = x_orig[np.newaxis, ...]
        x_adv   = deepfool.generate(x=x_input)
        x_test_adv_deepfool.append(x_adv[0])

        pred_idx = int(np.argmax(classifier_nn1.predict(x_adv), axis=1)[0])
        y_test_adv_deepfool.append(LABELS[pred_idx])

    # ricompone in tensor
    x_test_adv_deepfool = np.stack(x_test_adv_deepfool)
    y_test_adv_deepfool = np.array(y_test_adv_deepfool)
    
    # Save the adversarial examples and labels
    torch.save(x_test_adv_deepfool, deepfool_adv_folder + '/x_test_adv_deepfool_g.pt')
    torch.save(y_test_adv_deepfool, deepfool_adv_folder +'/y_test_adv_deepfool_g.pt')

Adversarial samples and their predicted labels are loaded from disk, and standard metrics are computed. These include overall classification accuracy and a breakdown of correct vs. incorrect predictions using `print_basic_metrics`.

To support visual inspection, `plot_predicted_images` is used to display clean and adversarial examples side by side, highlighting changes in model predictions.

In [ ]:
# ————————————————————————————————————————————————————
# DeepFool Error Generic (with sample selection + plot)
# ————————————————————————————————————————————————————

x_test_adv_deepfool = torch.load(deepfool_adv_folder + '/x_test_adv_deepfool_g.pt')
y_test_adv_deepfool = torch.load(deepfool_adv_folder + '/y_test_adv_deepfool_g.pt')

deepfool_epsilon      = 1e-4
deepfool_max_iter     = 1

print("=== Deepfool Adversarial Evaluation (Error Generic) ===")
print(f"ε = {deepfool_epsilon}, iter = {deepfool_max_iter}")

# Metriche generali
correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_deepfool,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_deepfool
)
acc = accuracy_score(y_true, y_test_adv_deepfool)
print(f"Accuracy           : {acc*100:.2f}%")

# === Plot + salvataggio ===
plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_deepfool,
    y_true         = y_true,
    y_pred         = y_test_adv_deepfool,
    y_pred_adv     = y_test_adv_deepfool,
    title          = f"DeepFool (Error Generic) - ε={deepfool_epsilon}, iter={deepfool_max_iter}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = deepfool_img_path
)

Two separate curves are then computed to analyse the impact of:

1. **Perturbation Threshold (`ε`)**:
    By increasing the allowed perturbation magnitude, the attack becomes more effective. This experiment keeps the number of iterations fixed and observes how accuracy drops as ε increases. It provides insight into how sensitive the model is to even small changes in the input.

2. **Number of Iterations (`max_iter`)**:
    With a fixed ε, this setup explores how the number of attack steps affects the success of the adversarial manipulation. As DeepFool is iterative in nature, increasing the number of iterations can lead to more precise boundary crossings and stronger attacks.

For each configuration, adversarial examples are generated individually for each sample, and predictions are evaluated using standard accuracy and `print_basic_metrics`. The results are plotted using `plot_multiple_sec_curves`.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - DeepFool (Error Generic)
# —————————————————————————————————————————————————————————————

eps_values       = [1e-5, 1e-4, 5e-4, 1e-3, 5e-3]
max_iter_values  = [1, 3, 5, 7, 10]

epsilon          = 1e-4
max_iter         = 1

curve_colors = ['crimson', 'blue']

# 1) Accuracy vs ε (fisso max_iter)
accuracies_eps   = [0.98]

print("=== SEC: DeepFool vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples with epsilon = {eps:.5f}, max_iter = {max_iter}")
    deepfool = DeepFool(
        classifier=classifier_nn1,
        epsilon=eps,
        max_iter=max_iter,
        verbose=False
    )
    
    x_adv_deepfool = []
    y_pred_adv_deepfool = []
    
    for i, x_orig in enumerate(tqdm(x_test_aligned_nn1, desc=f"DeepFool ε={eps:.0e}", leave=False)):
        x_input = x_orig[np.newaxis, ...]
        x_adv   = deepfool.generate(x=x_input)
        x_adv_deepfool.append(x_adv[0])

        pred_idx = int(np.argmax(classifier_nn1.predict(x_adv), axis=1)[0])
        y_pred_adv_deepfool.append(LABELS[pred_idx])

    # ricompone in tensor
    x_adv_deepfool = np.stack(x_adv_deepfool)
    y_pred_adv_deepfool = np.array(y_pred_adv_deepfool)
    
    correct, incorrect = print_basic_metrics(
        y_true,
        y_pred_adv_deepfool,
        x_orig = x_test_aligned_nn1,
        x_adv  = x_adv_deepfool
    )
    acc = accuracy_score(y_true, y_pred_adv_deepfool)
    accuracies_eps.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv_deepfool, x_test_aligned_nn1, x_adv_deepfool)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, accuracies_eps, 'o', curve_colors[0], 'Accuracy')
    ],
    title=f"Accuracy vs Epsilon (iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# 2) Accuracy vs max_iter (fisso epsilon)
accuracies_iter  = [0.98]

print("\n=== SEC: DeepFool vs max_iter ===")
for n_iter in max_iter_values:
    print(f"\n→ Generating adversarial examples with max_iter = {n_iter}, epsilon = {epsilon:.5f}")
    deepfool = DeepFool(
        classifier=classifier_nn1,
        epsilon=epsilon,
        max_iter=n_iter,
        verbose=False
    )
    
    x_adv_deepfool = []
    y_pred_adv_deepfool = []
    
    for i, x_orig in enumerate(tqdm(x_test_aligned_nn1, desc=f"DeepFool iter={n_iter}", leave=False)):
        x_input = x_orig[np.newaxis, ...]                # shape (1, C, H, W)
        x_adv   = deepfool.generate(x=x_input)             # ritorna shape (1, C, H, W)
        x_adv_deepfool.append(x_adv[0])             # prendi il tensor 0

        pred_idx = int(np.argmax(classifier_nn1.predict(x_adv), axis=1)[0])
        y_pred_adv_deepfool.append(LABELS[pred_idx])

    # ricompone in tensor
    x_adv_deepfool = np.stack(x_adv_deepfool)
    y_pred_adv_deepfool = np.array(y_pred_adv_deepfool)
    
    
    correct, incorrect = print_basic_metrics(
        y_true,
        y_pred_adv_deepfool,
        x_orig = x_test_aligned_nn1,
        x_adv  = x_adv_deepfool
    )
    acc = accuracy_score(y_true, y_pred_adv_deepfool)
    accuracies_iter.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred_adv_deepfool, x_test_aligned_nn1, x_adv_deepfool)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, accuracies_iter, 's', curve_colors[1], 'Accuracy')
    ],
    title=f"Accuracy vs max_iter (ε={epsilon})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.savefig(deepfool_img_path, bbox_inches='tight')
plt.show()

### Carlini Wagner (CW) $L_{\infty}$ - Adversarial Attack

We conclude the evaluation of adversarial robustness with the `Carlini & Wagner L_inf attack`, a powerful optimisation-based method known for its effectiveness against even robustly trained models. Unlike earlier attacks that rely on direct gradient steps or minimal perturbation strategies, CW $L_{\infty}$ formulates the adversarial objective as a constrained optimisation problem, allowing it to produce subtle but highly effective perturbations.

The $L_{\infty}$ variant enforces a strict bound on the maximum change allowed per pixel, ensuring that the adversarial perturbations remain imperceptible while still achieving misclassification. This makes it particularly relevant in practical adversarial scenarios where stealth is critical.

To support this final evaluation, the required folder structure is defined: a directory for saving the generated adversarial examples and a results folder with separate subfolders for the error generic and error specific configurations.

In [ ]:
from art.attacks.evasion import CarliniLInfMethod

cw_generate = True

cw_adv_folder = os.path.join(adversarial_folder, 'CW_L_inf_nn1')
os.makedirs(cw_adv_folder, exist_ok=True)
cw_results_folder = os.path.join(attack_folder, 'CW_results')
os.makedirs(cw_results_folder, exist_ok=True)

cw_inf_error_gen_folder    = os.path.join(cw_results_folder, "CW_Error_Generic_L_inf")
os.makedirs(cw_inf_error_gen_folder, exist_ok=True)
cw_inf_error_spec_folder     = os.path.join(cw_results_folder, "CW_Error_Specific")
os.makedirs(cw_inf_error_spec_folder, exist_ok=True)

cw_inf_img_path_g  = os.path.join(cw_inf_error_gen_folder, "cw_error_generic_plot.png")
cw_img_path_s = os.path.join(cw_inf_error_spec_folder, "cw_error_specific_plot.png")

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

As part of this final stage in the robustness evaluation, this block performs the generation of adversarial examples using the Carlini & Wagner L∞ method in the `error generic setting`. The aim here is to cause misclassification of the inputs without targeting a specific class, while keeping the perturbations constrained under the L∞ norm.

The attack is configured with the following parameters:

- `learning_rate = 0.01`

- `max_iter = 7`

- `initial_const = 1e-5`

- `largest_const = 1.0`

- `const_factor = 2.0`

- `decrease_factor = 0.9`

- `batch_size = 8`

These parameters govern the internal optimisation process, where the attack dynamically adjusts the constraint constant to find the minimal distortion required to induce misclassification. This iterative approach enables the method to fine-tune perturbations and produce effective adversarial examples with high confidence and minimal visibility.

Once generated, predictions are obtained using the NN1 model, and both the adversarial samples and their associated labels are saved to disk. 

In [ ]:
# ————————————————————————————————
# CW L∞ (Error Generic)
# ————————————————————————————————

# Parametri CW L∞
# cw_confidence     = 0.0 impostato a 0 di default dalla Classe CarliniLInfMethod
cw_learning_rate  = 0.01
cw_max_iter       = 7
cw_decrease_factor = 0.9
cw_initial_const  = 1e-5
cw_largest_const  = 1.0
cw_const_factor   = 2.0
cw_batch_size     = 8

if cw_generate:

    cw_inf = CarliniLInfMethod(
        classifier       = classifier_nn1,
        targeted         = False,
        learning_rate    = cw_learning_rate,
        max_iter         = cw_max_iter,
        decrease_factor  = cw_decrease_factor,
        initial_const    = cw_initial_const,
        largest_const    = cw_largest_const,
        const_factor     = cw_const_factor,
        batch_size       = cw_batch_size,
        verbose          = True         
    )

    x_test_adv_cw_g = cw_inf.generate(x=x_test_aligned_nn1)
    y_test_adv_cw_g = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_cw_g), axis=1)]

    if not os.path.exists(cw_adv_folder):
        os.makedirs(cw_adv_folder)

    torch.save(x_test_adv_cw_g, cw_adv_folder + '/x_test_adv_cw_g.pt')
    torch.save(y_test_adv_cw_g, cw_adv_folder + '/y_test_adv_cw_g.pt')

Using adversarial examples previously generated and stored, the model’s performance is assessed in terms of classification accuracy and prediction reliability under attack.

In [ ]:
x_test_adv_cw_g = torch.load(cw_adv_folder + '/x_test_adv_cw_g.pt')
y_test_adv_cw_g = torch.load(cw_adv_folder + '/y_test_adv_cw_g.pt')

cw_learning_rate  = 0.01
cw_max_iter       = 7
cw_decrease_factor = 0.9
cw_initial_const  = 1e-5
cw_largest_const  = 1.0
cw_const_factor   = 2.0
cw_batch_size     = 8

print("=== CW L∞ Adversarial Evaluation (Error Generic) ===")
print(f"Confidence = 0.0, max_iter = {cw_max_iter}, learning_rate = {cw_learning_rate}, decrease_factor = {cw_decrease_factor}\n")
correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_cw_g,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_cw_g
)
acc = accuracy_score(y_true, y_test_adv_cw_g)
print(f"Accuracy           : {acc*100:.2f}%")

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_cw_g,
    y_true         = y_true,
    y_pred         = y_test_adv_cw_g,
    y_pred_adv     = y_test_adv_cw_g,
    title          = f"CW L∞ (Error Generic) - Confidence = 0.0, max_iter = {cw_max_iter}, learning_rate = {cw_learning_rate}, decrease_factor = {cw_decrease_factor}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = cw_inf_img_path_g
)

#### Error Specific

In this last step, adversarial examples are generated using the Carlini & Wagner L∞ attack in its `targeted configuration`.

The procedure begins by calling the usual helper function `generate_one_hot_target_and_plot_image` that both generates the required one-hot encoded target labels for the attack and displays a clean reference image of the chosen target class.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

The attack is configured with a target `confidence` of `0.5`, meaning the optimisation will push the output logits beyond this margin to enforce the target prediction. Additional parameters — such as `learning_rate = 0.01`, `max_iter = 7`, and a gradually reduced constraint via `decrease_factor = 0.9` — control the convergence and aggressiveness of the optimisation process. These settings balance efficiency with stealth, ensuring the perturbations remain within the L∞ norm while maintaining adversarial effectiveness.

In [ ]:
cw_target_conf     = 0.5
cw_learning_rate   = 0.01
cw_max_iter        = 7
cw_initial_const   = 1e-5
cw_largest_const   = 1.0
cw_const_factor    = 0.5
cw_decrease_factor = 0.9
cw_batch_size      = 8


if cw_generate:
    attack = CarliniLInfMethod(
        classifier       = classifier_nn1,
        confidence       = cw_target_conf,
        targeted         = True,
        learning_rate    = cw_learning_rate,
        max_iter         = cw_max_iter,
        decrease_factor  = cw_decrease_factor,
        initial_const    = cw_initial_const,
        largest_const    = cw_largest_const,
        const_factor     = cw_const_factor,
        batch_size       = cw_batch_size,
        verbose          = True         
    )

    one_hot = to_categorical([target_class], nb_classes=len(LABELS))[0]

    one_hot_targeted_label = np.tile(one_hot, (len(x_test_aligned_nn1), 1))
    
    x_test_adv_cw_s = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)
    
    y_test_adv_cw_s = LABELS[np.argmax(classifier_nn1.predict(x_test_adv_cw_s), axis=1)]
    
    if not os.path.exists(cw_adv_folder):
        os.makedirs(cw_adv_folder)
        
    # Save the adversarial examples and labels
    torch.save(x_test_adv_cw_s, cw_adv_folder + '/x_test_adv_cw_s.pt')
    torch.save(y_test_adv_cw_s, cw_adv_folder +'/y_test_adv_cw_s.pt')

The adversarial examples and predictions are loaded from disk, and two key metrics are computed: overall accuracy and the targeted success rate, which quantifies how often the model's output matches the intended identity. The results are displayed alongside a visual comparison using plot_predicted_images, offering a direct look at the effect of the perturbations on model behaviour.

In [ ]:
# —————————————————————————————————————————————————————————————
# CW L_inf Error Specific (Targeted)
# —————————————————————————————————————————————————————————————

x_test_adv_cw_s = torch.load(cw_adv_folder + '/x_test_adv_cw_s.pt')
y_test_adv_cw_s = torch.load(cw_adv_folder + '/y_test_adv_cw_s.pt')

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

acc = accuracy_score(y_true, y_test_adv_cw_s)
sr  = (np.array(y_test_adv_cw_s) == target_name).mean() * 100

cw_target_conf     = 0.5
cw_learning_rate   = 0.01
cw_max_iter        = 7
cw_initial_const   = 1e-5
cw_largest_const   = 1.0
cw_const_factor    = 0.5
cw_decrease_factor = 0.9
cw_batch_size      = 8

print("=== CW L∞ Error Specific (Targeted) ===")
print(f"Target: {target_name} | Confidence: {cw_target_conf} | Iter: {cw_max_iter}\n")
print(f"Accuracy           : {acc*100:.2f}%")
print(f"Targeted Success   : {sr:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true,
    y_test_adv_cw_s,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_cw_s
)

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_adv_cw_s,
    y_true         = y_true,
    y_pred_adv     = y_test_adv_cw_s,
    y_pred         = y_test_adv_cw_s,
    title          = f"CW L∞ Error Specific - Target: {target_name}",
    id_test_images = idx_test_images,
    image_idx      = 0,
    save_path      = cw_img_path_s
)